In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 1 — Environment Setup & Imports
# ═══════════════════════════════════════════════════════════════════
!pip install -q transformers datasets accelerate

import math, re, copy, time
from typing import Optional, List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

torch.manual_seed(42)
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print(f"Device  : {device}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU     : {p.name}")
    print(f"VRAM    : {p.total_memory / 2**30:.1f} GB")
print(f"AMP     : {USE_AMP}")

Device  : cuda
GPU     : Tesla T4
VRAM    : 14.6 GB
AMP     : True


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 2 — ERD Model Configuration
#  Tuned for Colab T4 (15 GB VRAM).  Double d_model/n_layers
#  for Colab Pro (A100 40 GB).
# ═══════════════════════════════════════════════════════════════════

@dataclass
class ERDConfig:
    # ── Vocabulary ─────────────────────────────────────────────────
    vocab_size   : int   = 32000
    max_seq_len  : int   = 1024

    # ── Backbone ───────────────────────────────────────────────────
    d_model      : int   = 512    # residual width
    n_layers     : int   = 8
    n_heads      : int   = 8
    head_dim     : int   = 64     # d_model must equal n_heads × head_dim

    # ── Phase 1B — Multi-Head Latent Attention (MLA) ───────────────
    kv_lora_rank : int   = 128    # vs standard 2×8×64=1024  → 87.5% savings
    q_lora_rank  : int   = 256
    rope_theta   : float = 10_000.0

    # ── Phase 1C — Mixture of Experts (MoE) ────────────────────────
    n_routed_exp : int   = 8      # selectable experts
    n_shared_exp : int   = 1      # always-active expert
    top_k        : int   = 2      # experts per token → 25% FFN compute
    d_ff_routed  : int   = 1024   # per routed expert
    d_ff_shared  : int   = 512    # shared expert

    # ── Misc ────────────────────────────────────────────────────────
    rms_eps      : float = 1e-6
    balance_coef : float = 0.01   # load-balance aux loss weight

    def __post_init__(self):
        assert self.d_model == self.n_heads * self.head_dim, \
            f"d_model ({self.d_model}) ≠ n_heads × head_dim"
        kv_std   = 2 * self.n_heads * self.head_dim
        saving   = 100 * (1 - self.kv_lora_rank / kv_std)
        act_pct  = 100 * self.top_k / self.n_routed_exp
        print(f"KV-cache compression : {self.kv_lora_rank}/{kv_std} → {saving:.0f}% savings")
        print(f"FFN active compute   : top-{self.top_k}/{self.n_routed_exp} → {act_pct:.0f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 3 — RMSNorm + Rotary Positional Embeddings
# ═══════════════════════════════════════════════════════════════════

class RMSNorm(nn.Module):
    """Root-Mean-Square LayerNorm — no mean-centering, ~10% faster."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).to(x.dtype) * self.weight


class RotaryEmbedding(nn.Module):
    """
    RoPE — positions are encoded as rotations applied to Q and K vectors.
    No learnable parameters; relative position falls out automatically.
    Cache is pre-built and auto-extended if needed.
    """
    def __init__(self, dim: int, max_seq_len: int = 4096, theta: float = 10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq_len)

    def _build_cache(self, length: int):
        t     = torch.arange(length, device=self.inv_freq.device,
                             dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)          # (L, dim/2)
        emb   = torch.cat([freqs, freqs], dim=-1)      # (L, dim)
        # shapes: (1, 1, L, dim) — broadcast over batch & heads
        self.register_buffer("cos_cached", emb.cos()[None, None])
        self.register_buffer("sin_cached", emb.sin()[None, None])

    def forward(self, seq_len: int, device: torch.device):
        if seq_len > self.cos_cached.shape[2]:          # auto-extend
            self._build_cache(seq_len * 2)
        return (
            self.cos_cached[:, :, :seq_len].to(device),
            self.sin_cached[:, :, :seq_len].to(device),
        )


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(
    q: torch.Tensor, k: torch.Tensor,
    cos: torch.Tensor, sin: torch.Tensor,
    q_offset: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Apply RoPE to Q (starting at q_offset) and K (always from position 0).
    q  shape: (B, H, T_q,  D_h)
    k  shape: (B, H, T_kv, D_h)
    cos/sin : (1, 1, T_kv, D_h)
    """
    T_q  = q.shape[2]
    T_kv = k.shape[2]
    q_cos = cos[:, :, q_offset : q_offset + T_q]
    q_sin = sin[:, :, q_offset : q_offset + T_q]
    k_cos = cos[:, :, :T_kv]
    k_sin = sin[:, :, :T_kv]
    q_rot = (q * q_cos) + (rotate_half(q) * q_sin)
    k_rot = (k * k_cos) + (rotate_half(k) * k_sin)
    return q_rot, k_rot

print("✅  Cell 3 OK — RMSNorm, RotaryEmbedding, apply_rope")

✅  Cell 3 OK — RMSNorm, RotaryEmbedding, apply_rope


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 4 — Multi-Head Latent Attention (MLA)
#
#  Standard KV-cache  →  2 × n_heads × head_dim floats per token
#  MLA KV-cache       →  kv_lora_rank floats per token
#
#  At seq_len=1024, n_layers=8, n_heads=8, head_dim=64:
#    Standard : 1024 × 8 × (2×8×64) × 2B (bf16)  ≈ 16 MB
#    MLA      : 1024 × 8 × 128      × 2B (bf16)  ≈  2 MB  (87.5% savings)
# ═══════════════════════════════════════════════════════════════════

class MultiHeadLatentAttention(nn.Module):
    def __init__(self, cfg: ERDConfig):
        super().__init__()
        self.H   = cfg.n_heads
        self.D_h = cfg.head_dim
        self.d_c = cfg.kv_lora_rank   # latent KV dimension (what gets cached)

        # Q path: compress  →  decompress
        self.W_dq   = nn.Linear(cfg.d_model, cfg.q_lora_rank,       bias=False)
        self.W_uq   = nn.Linear(cfg.q_lora_rank, self.H * self.D_h, bias=False)
        self.q_norm = RMSNorm(self.D_h, cfg.rms_eps)

        # KV path: compress (cached!)  →  decompress per-forward
        self.W_dkv  = nn.Linear(cfg.d_model, self.d_c,              bias=False)
        self.W_uk   = nn.Linear(self.d_c, self.H * self.D_h,        bias=False)
        self.k_norm = RMSNorm(self.D_h, cfg.rms_eps)
        self.W_uv   = nn.Linear(self.d_c, self.H * self.D_h,        bias=False)

        # Output projection
        self.W_o    = nn.Linear(self.H * self.D_h, cfg.d_model,     bias=False)

        self.rope   = RotaryEmbedding(self.D_h, cfg.max_seq_len, cfg.rope_theta)

    def forward(
        self,
        x         : torch.Tensor,
        kv_cache  : Optional[torch.Tensor] = None,
        use_cache : bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, T, _ = x.shape

        # ── Queries: compress → decompress ──────────────────────────
        q = self.W_uq(self.W_dq(x))                           # (B, T, H·D_h)
        q = q.view(B, T, self.H, self.D_h).transpose(1, 2)    # (B, H, T, D_h)
        q = self.q_norm(q)

        # ── KV latent — only this tensor is cached during generation ─
        c_kv = self.W_dkv(x)                                   # (B, T, d_c)
        if kv_cache is not None:
            c_kv = torch.cat([kv_cache, c_kv], dim=1)          # (B, T_prev+T, d_c)
        new_cache = c_kv if use_cache else None
        T_kv = c_kv.shape[1]

        # ── Keys & Values: decompressed fresh each forward ───────────
        k = self.W_uk(c_kv).view(B, T_kv, self.H, self.D_h).transpose(1, 2)
        k = self.k_norm(k)
        v = self.W_uv(c_kv).view(B, T_kv, self.H, self.D_h).transpose(1, 2)

        # ── RoPE: queries start at offset = T_kv - T ─────────────────
        cos, sin  = self.rope(T_kv, x.device)
        q_offset  = T_kv - T
        q, k      = apply_rope(q, k, cos, sin, q_offset=q_offset)

        # ── Scaled dot-product attention (uses Flash Attention if available)
        #    is_causal=True only during prefill (no cache, multiple tokens)
        is_causal = (kv_cache is None) and (T > 1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.H * self.D_h)
        return self.W_o(out), new_cache

print("✅  Cell 4 OK — MultiHeadLatentAttention")

✅  Cell 4 OK — MultiHeadLatentAttention


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 5 — Fine-Grained Mixture of Experts with Shared Experts
#
#  Total expert capacity  : n_routed_exp × d_ff_routed params
#  Active per token       : n_shared_exp × d_ff_shared
#                         + top_k × d_ff_routed
#  At top_k=2, n=8  →  25% of routed FFN compute per token.
#  The model has the *knowledge* of an 8× larger FFN.
# ═══════════════════════════════════════════════════════════════════

class SwiGLUFFN(nn.Module):
    """Single expert — SwiGLU gated feed-forward (same as Llama/Qwen)."""
    def __init__(self, d_in: int, d_ff: int):
        super().__init__()
        self.gate = nn.Linear(d_in, d_ff, bias=False)
        self.up   = nn.Linear(d_in, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_in, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))


class MixtureOfExperts(nn.Module):
    def __init__(self, cfg: ERDConfig):
        super().__init__()
        self.n_routed = cfg.n_routed_exp
        self.top_k    = cfg.top_k

        # Shared experts — every token always passes through these
        self.shared = nn.ModuleList([
            SwiGLUFFN(cfg.d_model, cfg.d_ff_shared)
            for _ in range(cfg.n_shared_exp)
        ])

        # Routed experts — only top_k activated per token
        self.routed = nn.ModuleList([
            SwiGLUFFN(cfg.d_model, cfg.d_ff_routed)
            for _ in range(cfg.n_routed_exp)
        ])

        # Router: linear score per expert, no bias
        self.router = nn.Linear(cfg.d_model, cfg.n_routed_exp, bias=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, D = x.shape
        xf      = x.reshape(-1, D)                          # (N, D)

        # ── Shared experts (unconditional path) ─────────────────────
        shared_out = torch.zeros_like(xf)
        for exp in self.shared:
            shared_out = shared_out + exp(xf)

        # ── Router ──────────────────────────────────────────────────
        logits  = self.router(xf)                           # (N, n_routed)
        probs   = F.softmax(logits, dim=-1)                 # (N, n_routed)
        topk_w, topk_i = probs.topk(self.top_k, dim=-1)    # (N, top_k)
        topk_w  = topk_w / (topk_w.sum(-1, keepdim=True) + 1e-9)  # renorm

        # ── Routed experts (conditional path) ───────────────────────
        routed_out = torch.zeros_like(xf)
        for e in range(self.n_routed):
            # mask2d[n, k] = True if token n chose expert e at slot k
            mask2d     = (topk_i == e)                      # (N, top_k)
            token_mask = mask2d.any(-1)                     # (N,)
            if not token_mask.any():
                continue
            # combine weights across all k-slots that selected expert e
            weight     = (topk_w * mask2d.float()).sum(-1)  # (N,)
            expert_out = self.routed[e](xf[token_mask])     # (n_sel, D)
            routed_out[token_mask] += weight[token_mask, None] * expert_out

        # ── Auxiliary load-balance loss ──────────────────────────────
        # Minimised when all experts receive equal average probability.
        # Gradient only flows through `probs`, not through top-k selection.
        avg_prob     = probs.mean(0)                        # (n_routed,)
        balance_loss = self.n_routed * (avg_prob ** 2).sum()

        return (shared_out + routed_out).view(B, T, D), balance_loss

print("✅  Cell 5 OK — SwiGLUFFN, MixtureOfExperts")

✅  Cell 5 OK — SwiGLUFFN, MixtureOfExperts


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 6 — ERDBlock (one transformer layer) + ERDModel (full LM)
# ═══════════════════════════════════════════════════════════════════

class ERDBlock(nn.Module):
    """Pre-norm MLA  →  residual  →  pre-norm MoE  →  residual."""
    def __init__(self, cfg: ERDConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.attn      = MultiHeadLatentAttention(cfg)
        self.ffn_norm  = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.moe       = MixtureOfExperts(cfg)

    def forward(
        self,
        x        : torch.Tensor,
        kv_cache : Optional[torch.Tensor] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], torch.Tensor]:
        attn_out, new_cache = self.attn(self.attn_norm(x), kv_cache, use_cache)
        x = x + attn_out
        moe_out, balance   = self.moe(self.ffn_norm(x))
        x = x + moe_out
        return x, new_cache, balance


# ────────────────────────────────────────────────────────────────────

class ERDModel(nn.Module):
    """
    Efficient Reasoning & Distillation Language Model
    ─────────────────────────────────────────────────
    Phase 1 : this class  (MLA + MoE sparse backbone)
    Phase 2 : GRPO trainer wraps it   (Cell 10)
    Phase 3 : distillation functions  (Cell 12)
    """

    def __init__(self, cfg: ERDConfig):
        super().__init__()
        self.cfg     = cfg
        self.embed   = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers  = nn.ModuleList([ERDBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # Weight tying: embedding and LM-head share one matrix (saves ~32 M params)
        self.lm_head.weight = self.embed.weight
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    # ── Forward ─────────────────────────────────────────────────────
    def forward(
        self,
        input_ids : torch.Tensor,
        labels    : Optional[torch.Tensor] = None,
        kv_caches : Optional[List[Optional[torch.Tensor]]] = None,
        use_cache : bool = False,
    ) -> Dict[str, torch.Tensor]:

        x = self.embed(input_ids)
        new_caches    = []
        total_balance = torch.zeros((), device=x.device)

        for i, layer in enumerate(self.layers):
            cache = kv_caches[i] if kv_caches else None
            x, nc, bal = layer(x, cache, use_cache)
            if use_cache:
                new_caches.append(nc)
            total_balance = total_balance + bal

        x      = self.norm(x)
        logits = self.lm_head(x)                           # (B, T, vocab)

        out = {"logits": logits}
        if use_cache:
            out["kv_caches"] = new_caches

        if labels is not None:
            s_logits = logits[:, :-1].contiguous()
            s_labels = labels[:, 1:].contiguous()
            lm_loss  = F.cross_entropy(
                s_logits.view(-1, self.cfg.vocab_size),
                s_labels.view(-1), ignore_index=-100,
            )
            avg_bal          = total_balance / self.cfg.n_layers
            out["loss"]         = lm_loss + self.cfg.balance_coef * avg_bal
            out["lm_loss"]      = lm_loss
            out["balance_loss"] = avg_bal

        return out

    # ── Generation with KV-cache ─────────────────────────────────────
    @torch.no_grad()
    def generate(
        self,
        input_ids      : torch.Tensor,
        max_new_tokens : int   = 256,
        temperature    : float = 0.8,
        top_p          : float = 0.9,
        eos_token_id   : Optional[int] = None,
    ) -> torch.Tensor:
        self.eval()
        generated = input_ids.clone()
        kv_caches = [None] * self.cfg.n_layers

        for step in range(max_new_tokens):
            # First step: prefill full prompt; subsequent steps: one token
            cur = generated if step == 0 else generated[:, -1:]
            out = self.forward(cur, kv_caches=kv_caches, use_cache=True)
            kv_caches = out["kv_caches"]

            next_logits = out["logits"][:, -1, :] / max(temperature, 1e-8)
            probs       = F.softmax(next_logits, dim=-1)

            # Nucleus (top-p) sampling
            s_probs, s_idx = probs.sort(descending=True)
            cumsum         = s_probs.cumsum(-1)
            nuke_mask      = (cumsum - s_probs) > top_p
            s_probs[nuke_mask] = 0.0
            s_probs /= s_probs.sum(-1, keepdim=True).clamp(min=1e-9)
            next_tok = s_idx.gather(-1, torch.multinomial(s_probs, 1))

            generated = torch.cat([generated, next_tok], dim=-1)
            if eos_token_id is not None and (next_tok == eos_token_id).all():
                break

        return generated

    # ── Utilities ────────────────────────────────────────────────────
    def count_params(self) -> Tuple[int, int]:
        total = sum(p.numel() for p in self.parameters())
        # Inactive experts don't participate in each forward
        inactive_per_layer = (
            sum(p.numel() for p in self.layers[0].moe.routed[0].parameters())
            * (self.cfg.n_routed_exp - self.cfg.top_k)
        )
        active = total - inactive_per_layer * self.cfg.n_layers
        return total, active

print("✅  Cell 6 OK — ERDBlock, ERDModel")

✅  Cell 6 OK — ERDBlock, ERDModel


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 7 — Instantiate and Verify the Phase-1 Architecture
# ═══════════════════════════════════════════════════════════════════

config = ERDConfig()
model  = ERDModel(config).to(device)

total, active = model.count_params()
print(f"\nTotal parameters   : {total/1e6:.1f} M")
print(f"Active per forward : {active/1e6:.1f} M  ({100*active/total:.0f}%)")

# ── Smoke test: forward pass ─────────────────────────────────────────
B, T   = 2, 64
ids    = torch.randint(0, config.vocab_size, (B, T), device=device)
with torch.no_grad():
    out = model(ids, labels=ids)

expected_loss = math.log(config.vocab_size)   # random-init baseline
print(f"\n✅  Forward pass OK")
print(f"   logits       : {out['logits'].shape}")
print(f"   lm_loss      : {out['lm_loss'].item():.4f}  (random ≈ {expected_loss:.2f})")
print(f"   balance_loss : {out['balance_loss'].item():.4f}")

# ── KV-cache size comparison ──────────────────────────────────────────
SEQ = config.max_seq_len
bytes_std = 2 * config.n_heads * config.head_dim * SEQ * config.n_layers * 2  # BF16
bytes_mla = config.kv_lora_rank                * SEQ * config.n_layers * 2
print(f"\n📉  KV-cache at seq_len={SEQ}, n_layers={config.n_layers}")
print(f"   Standard MHA : {bytes_std/2**20:.1f} MB")
print(f"   MLA (ours)   : {bytes_mla/2**20:.1f} MB  ({100*bytes_mla/bytes_std:.1f}%)")

if torch.cuda.is_available():
    print(f"\n   VRAM used    : {torch.cuda.memory_allocated()/2**20:.0f} MB")

KV-cache compression : 128/1024 → 88% savings
FFN active compute   : top-2/8 → 25%

Total parameters   : 129.1 M
Active per forward : 53.7 M  (42%)

✅  Forward pass OK
   logits       : torch.Size([2, 64, 32000])
   lm_loss      : 10.5234  (random ≈ 10.37)
   balance_loss : 1.0049

📉  KV-cache at seq_len=1024, n_layers=8
   Standard MHA : 16.0 MB
   MLA (ours)   : 2.0 MB  (12.5%)

   VRAM used    : 1033 MB


In [ ]:
!pip install -q --upgrade torchao peft

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  IMPROVEMENT CELL 1 — Cold Start SFT (FULLY SELF-CONTAINED)
#  Run this directly — no other cell needed first
# ═══════════════════════════════════════════════════════════════════

import os, gc, re, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Step 1: Load tokenizer + model ───────────────────────────────────
BASE_ID   = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, torch_dtype=torch.float16, device_map="auto"
)
lora_cfg = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16, lora_alpha     = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout   = 0.0, bias = "none",
)
model = get_peft_model(model, lora_cfg)
dev   = next(model.parameters()).device
print(f"✅ Model ready. VRAM: {torch.cuda.memory_allocated()/2**30:.1f} GB")

# ── Step 2: Load dataset ─────────────────────────────────────────────
gsm8k      = load_dataset("openai/gsm8k", "main")
train_data = gsm8k["train"]

# ── Step 3: Helper functions ─────────────────────────────────────────
SYSTEM = (
    "You are a math reasoning assistant. "
    "Show all working inside <think></think> tags, "
    "then give ONLY the final integer inside <answer></answer> tags.\n"
)

def make_prompt(q):
    return f"{SYSTEM}\nProblem: {q}\n\n<think>"

def extract_gsm8k_answer(s):
    m = re.search(r'####\s*(-?[\d,]+)', s)
    return m.group(1).replace(',', '') if m else None

def extract_model_answer(s):
    m = re.search(r'<answer>\s*(-?[\d,]+)\s*</answer>', s)
    return m.group(1).replace(',', '') if m else None

# ── Step 4: Build Cold Start examples ────────────────────────────────
def format_cold_start_example(question, solution):
    clean  = re.sub(r'<<[^>]+>>', '', solution)
    parts  = clean.split('####')
    if len(parts) < 2: return None
    reasoning = parts[0].strip()
    answer    = parts[1].strip().replace(',','').replace(' ','')
    if not answer.lstrip('-').isdigit(): return None
    completion = f"{reasoning}</think><answer>{answer}</answer>"
    return {
        'prompt'     : make_prompt(question),
        'completion' : completion,
        'answer'     : answer
    }

cold_start_raw = []
for item in train_data.select(range(500)):
    ex = format_cold_start_example(item['question'], item['answer'])
    if ex: cold_start_raw.append(ex)

print(f"Cold start examples built: {len(cold_start_raw)}")
print(f"Sample: {cold_start_raw[0]['completion'][:100]}...")

# ── Step 5: SFT Dataset ───────────────────────────────────────────────
class ColdStartDataset(Dataset):
    def __init__(self, data, tok, max_len=350):
        self.data = data; self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item      = self.data[idx]
        full_text = item['prompt'] + item['completion']
        enc = self.tok(
            full_text, max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )
        input_ids = enc['input_ids'].squeeze(0)
        prompt_len = len(self.tok(
            item['prompt'], add_special_tokens=False
        )['input_ids'])
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        return {
            'input_ids'      : input_ids,
            'labels'         : labels,
            'attention_mask' : enc['attention_mask'].squeeze(0)
        }

# ── Step 6: SFT Training ──────────────────────────────────────────────
def run_cold_start_sft(model, tok, data, n_epochs=3, lr=2e-5, batch_size=2):
    dataset   = ColdStartDataset(data, tok)
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    dev       = next(model.parameters()).device
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt       = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.01)

    print(f"\n{'─'*54}")
    print(f"  Cold Start SFT | {len(data)} examples | {n_epochs} epochs")
    print(f"{'─'*54}")

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, steps = 0.0, 0
        for batch in loader:
            ids  = batch['input_ids'].to(dev)
            lbls = batch['labels'].to(dev)
            mask = batch['attention_mask'].to(dev)
            out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
            loss = out.loss
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item(); steps += 1
        print(f"  Epoch {epoch}/{n_epochs}  sft_loss {total/steps:.4f}")

    print(f"\n✅ Cold Start SFT complete")
    return model

model = run_cold_start_sft(
    model, tokenizer, cold_start_raw,
    n_epochs=3, lr=2e-5, batch_size=2
)

# ── Step 7: Sanity check ──────────────────────────────────────────────
model.eval()
test_q = "Mark has 50 apples. He gives 15 to his friend. How many remain?"
ids    = tokenizer(make_prompt(test_q), return_tensors='pt').input_ids.to(dev)
with torch.no_grad():
    out = model.generate(
        ids, max_new_tokens=120, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
response = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
if "</answer>" in response:
    response = response[:response.index("</answer>") + len("</answer>")]

print(f"\nSanity check:")
print(f"Q: {test_q}")
print(f"A: {response}")
print(f"Parsed: {extract_model_answer(response)} | Gold: 35")

# ── Save key variables for next cells ────────────────────────────────
print("\n✅ All variables ready for Improvement Cell 2")
print(f"   model, tokenizer, dev, train_data, make_prompt,")
print(f"   extract_model_answer all defined ✓")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ Model ready. VRAM: 1.5 GB
Cold start examples built: 500
Sample: Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.</thi...

──────────────────────────────────────────────────────
  Cold Start SFT | 500 examples | 3 epochs
──────────────────────────────────────────────────────
  Epoch 1/3  sft_loss 1.4462
  Epoch 2/3  sft_loss 0.2127


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  Epoch 3/3  sft_loss 0.2029

✅ Cold Start SFT complete

Sanity check:
Q: Mark has 50 apples. He gives 15 to his friend. How many remain?
A: He has 50-15=45 apples</think><answer>45</answer>
Parsed: 45 | Gold: 35

✅ All variables ready for Improvement Cell 2
   model, tokenizer, dev, train_data, make_prompt,
   extract_model_answer all defined ✓


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  IMPROVEMENT CELL 2 — Enhanced GRPO (FULLY SELF-CONTAINED)
# ═══════════════════════════════════════════════════════════════════

import contextlib, re, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM

# ── Re-define GSM8KDataset and train_dataset ──────────────────────────
class GSM8KDataset(Dataset):
    def __init__(self, data, tok, max_len=220):
        self.data = data; self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tok(
            make_prompt(item['question']),
            max_length=self.max_len, truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'correct_answer' : extract_gsm8k_answer(item['answer']),
        }

train_dataset = GSM8KDataset(train_data, tokenizer)
print(f"✅ train_dataset ready: {len(train_dataset)} problems")

# ── SmartReward ───────────────────────────────────────────────────────
class SmartReward:
    def __init__(self, tok): self.tok = tok
    def __call__(self, ids, correct):
        if not correct: return 0.0
        text = self.tok.decode(ids, skip_special_tokens=True)
        r    = 0.0
        has_full  = bool(re.search(r'<think>.+?</think>', text, re.DOTALL))
        has_open  = '<think>' in text
        if has_full:    r += 0.3
        elif has_open:  r += 0.1
        else:           r -= 0.5
        pred = extract_model_answer(text)
        if pred == correct:    r += 1.0
        elif pred is not None: r -= 0.2
        elif correct in text:  r += 0.2
        else:                  r -= 0.3
        return float(r)

# ── GRPOTrainerHF ─────────────────────────────────────────────────────
class GRPOTrainerHF:
    def __init__(self, model, tok, G=4, kl_coef=0.01,
                 max_gen=90, lr=3e-5):
        self.model   = model; self.tok = tok
        self.G       = G; self.kl_coef = kl_coef
        self.max_gen = max_gen
        self.reward  = SmartReward(tok)
        self.dev     = next(model.parameters()).device

        print("Loading frozen reference...")
        self.ref = AutoModelForCausalLM.from_pretrained(
            BASE_ID, torch_dtype=torch.float16, device_map="auto"
        )
        for p in self.ref.parameters():
            p.requires_grad_(False)
        self.ref.eval()

        trainable = [p for p in model.parameters() if p.requires_grad]
        self.opt  = torch.optim.AdamW(
            trainable, lr=lr, betas=(0.9, 0.95), weight_decay=0.01
        )
        print(f"✅ Ready. VRAM: {torch.cuda.memory_allocated()/2**30:.1f} GB")

    def _log_prob(self, mdl, input_ids, prompt_len, no_grad=False):
        ctx = torch.no_grad() if no_grad else contextlib.nullcontext()
        with ctx:
            logits = mdl(input_ids=input_ids).logits.float()
        gen_logits = logits[:, prompt_len - 1:-1, :]
        gen_labels = input_ids[:, prompt_len:]
        lp = F.log_softmax(gen_logits, dim=-1)
        return lp.gather(-1, gen_labels.unsqueeze(-1)).squeeze(-1).mean()

    def step(self, input_ids, correct):
        prompt_len = input_ids.shape[1]
        self.model.eval()
        completions = []
        with torch.no_grad():
            for _ in range(self.G):
                gen = self.model.generate(
                    input_ids,
                    max_new_tokens = self.max_gen,
                    do_sample      = True,
                    temperature    = 0.9,
                    top_p          = 0.9,
                    pad_token_id   = self.tok.eos_token_id,
                )
                completions.append(gen[:, prompt_len:])

        rewards = [self.reward(c[0], correct) for c in completions]
        r_t     = torch.tensor(rewards, dtype=torch.float32)
        std     = r_t.std(unbiased=False)

        if std < 1e-6:
            return {
                'loss': 0.0, 'rewards': rewards,
                'mean_reward': float(r_t.mean()),
                'n_correct': 0, 'skipped': True
            }

        adv = (r_t - r_t.mean()) / (std + 1e-8)
        self.model.train()
        self.opt.zero_grad()
        losses = []

        for comp, a in zip(completions, adv.tolist()):
            full   = torch.cat([input_ids, comp], dim=1)[:, :512]
            log_p  = self._log_prob(self.model, full, prompt_len)
            ref_lp = self._log_prob(self.ref,   full, prompt_len,
                                    no_grad=True)
            losses.append(-a * log_p + self.kl_coef * (log_p - ref_lp))

        loss = torch.stack(losses).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in self.model.parameters() if p.requires_grad], 1.0
        )
        self.opt.step()

        return {
            'loss'        : loss.item(),
            'rewards'     : rewards,
            'mean_reward' : float(r_t.mean()),
            'n_correct'   : sum(1 for r in rewards if r > 0.8),
            'skipped'     : False,
        }

    def train(self, dataset, n_steps=150, log_every=20):
        loader = DataLoader(dataset, batch_size=1, shuffle=True)
        it     = iter(loader)
        log, skips = [], 0

        print(f"\n{'─'*54}")
        print(f"  Enhanced GRPO (post Cold-Start SFT) | {n_steps} steps")
        print(f"  Reward should start POSITIVE this time")
        print(f"{'─'*54}\n")

        for step in range(1, n_steps + 1):
            try:   batch = next(it)
            except StopIteration:
                it = iter(loader); batch = next(it)

            answer = batch['correct_answer'][0]
            if not answer: continue

            ids   = batch['input_ids'].to(self.dev)
            stats = self.step(ids, answer)

            if stats['skipped']:
                skips += 1; continue
            log.append(stats)

            if step % log_every == 0 and log:
                w    = log[-log_every:]
                avgr = sum(m['mean_reward'] for m in w) / len(w)
                argc = sum(m['n_correct']   for m in w) / len(w)
                print(f"  step {step:4d}/{n_steps}  "
                      f"loss {stats['loss']:6.3f}  "
                      f"avg_reward {avgr:+.3f}  "
                      f"correct {argc:.1f}/{self.G}  "
                      f"skips={skips}")
        return log


# ── Run Enhanced GRPO ─────────────────────────────────────────────────
trainer2 = GRPOTrainerHF(
    model   = model,
    tok     = tokenizer,
    G       = 4,
    kl_coef = 0.005,   # lower — model already well-formatted after SFT
    max_gen = 100,
    lr      = 1e-5,    # lower — fine adjustment on good SFT base
)

history2 = trainer2.train(
    train_dataset,
    n_steps   = 500,
    log_every = 100,
)

# ── Quick eval after GRPO ─────────────────────────────────────────────
print("\n" + "═"*54)
print("  TEACHER EVAL — post Cold-Start + Enhanced GRPO")
print("═"*54)

eval_qs = [
    ("Natalia sold 48 clips in April and half as many in May. Total?", "72"),
    ("6 boxes, 12 apples each. Sell 20. Remaining?",                   "52"),
    ("Tom $50, buys 3 books at $8 each. How much left?",               "26"),
    ("Train 60 mph for 2.5 hours. Miles travelled?",                   "150"),
]

model.eval()
correct = 0
for q, gold in eval_qs:
    ids = tokenizer(
        make_prompt(q), return_tensors='pt'
    ).input_ids.to(dev)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=150, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    resp = tokenizer.decode(
        out[0][ids.shape[1]:], skip_special_tokens=True
    )
    if "</answer>" in resp:
        resp = resp[:resp.index("</answer>") + len("</answer>")]
    pred   = extract_model_answer(resp)
    status = "✅" if pred == gold else "❌"
    print(f"{status} {q[:55]}...")
    print(f"   → {resp.strip()[:80]}")
    print(f"   Parsed: {pred} | Gold: {gold}\n")
    if pred == gold: correct += 1

print(f"Score: {correct}/{len(eval_qs)} = {100*correct/len(eval_qs):.0f}%")
print(f"Previous best: 50% — target: 75%+")

✅ train_dataset ready: 7473 problems
Loading frozen reference...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Ready. VRAM: 3.3 GB

──────────────────────────────────────────────────────
  Enhanced GRPO (post Cold-Start SFT) | 500 steps
  Reward should start POSITIVE this time
──────────────────────────────────────────────────────

  step  100/500  loss  0.018  avg_reward -0.362  correct 0.0/4  skips=17
  step  300/500  loss -0.050  avg_reward -0.354  correct 0.0/4  skips=56
  step  400/500  loss  0.001  avg_reward -0.381  correct 0.0/4  skips=79

══════════════════════════════════════════════════════
  TEACHER EVAL — post Cold-Start + Enhanced GRPO
══════════════════════════════════════════════════════
✅ Natalia sold 48 clips in April and half as many in May....
   → April: 48 clips
May: 48/2 = 24 clips
Total: 48+24 = 72 clips</think><answer>72</
   Parsed: 72 | Gold: 72

✅ 6 boxes, 12 apples each. Sell 20. Remaining?...
   → 6 boxes x 12 apples/box = 72 apples
20 sold = 72 - 20 = 52 apples</think><answer
   Parsed: 52 | Gold: 52

✅ Tom $50, buys 3 books at $8 each. How much left?...
   → He

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  IMPROVEMENT CELL 3 — Rejection Sampling Fine-Tuning (RST)
#
#  WHY: After GRPO, the model sometimes gets correct answers.
#  Rejection sampling keeps ONLY the correct chains and fine-tunes
#  again — solidifying correct reasoning patterns.
#  This is DeepSeek's Round 2 SFT stage.
# ═══════════════════════════════════════════════════════════════════
# Add at very top of Improvement Cell 3
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnAnswerClose(StoppingCriteria):
    def __init__(self, tok): self.tok = tok
    def __call__(self, input_ids, scores, **kwargs):
        recent = self.tok.decode(input_ids[0, -20:], skip_special_tokens=False)
        return "</answer>" in recent
def rejection_sample(model, tok, dataset, n_problems=200, G=8):
    """
    For each problem, generate G solutions.
    Keep only the ones that are CORRECT.
    Returns a list of (prompt, correct_completion) pairs.
    """
    dev     = next(model.parameters()).device
    stop    = StoppingCriteriaList([StopOnAnswerClose(tok)])
    accepted = []

    model.eval()
    print(f"\nRejection sampling {n_problems} problems (G={G} each)...")

    for idx in range(min(n_problems, len(dataset))):
        item   = dataset[idx]
        answer = item['correct_answer']
        if not answer: continue

        ids = item['input_ids'].unsqueeze(0).to(dev)

        for _ in range(G):
            with torch.no_grad():
                gen = model.generate(
                    ids, max_new_tokens=120, do_sample=True,
                    temperature=0.8, top_p=0.9,
                    pad_token_id=tok.eos_token_id,
                    stopping_criteria=stop,
                )
            completion = tok.decode(
                gen[0][ids.shape[1]:], skip_special_tokens=True
            )
            # Keep only correct solutions
            if extract_model_answer(completion) == answer:
                prompt = tok.decode(ids[0], skip_special_tokens=True)
                accepted.append({
                    'prompt'     : prompt,
                    'completion' : completion,
                    'answer'     : answer,
                })
                break  # one correct solution per problem is enough

        if (idx + 1) % 50 == 0:
            print(f"  Sampled {idx+1}/{n_problems}  |  "
                  f"Accepted: {len(accepted)} ({100*len(accepted)/(idx+1):.0f}%)")

    print(f"\n✅ Rejection sampling done: {len(accepted)} correct chains collected")
    return accepted


# Run rejection sampling
rst_data = rejection_sample(model, tokenizer, train_dataset,
                             n_problems=200, G=8)

# Fine-tune on correct chains (Round 2 SFT)
if len(rst_data) > 10:
    print(f"\nFine-tuning on {len(rst_data)} correct reasoning chains...")
    model = run_cold_start_sft(
        model, tokenizer, rst_data,
        n_epochs=2, lr=1e-5, batch_size=2
    )
else:
    print("Not enough correct samples — skip RST, continue to distillation")

NameError: name 'model' is not defined

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  IMPROVEMENT CELL 4 — Distil into Pretrained Student
#
#  WHY: Our ERDModel student started from RANDOM WEIGHTS (0% accuracy).
#  DeepSeek distils into a pretrained model (already knows language).
#  We use a fresh Qwen2.5-0.5B as student — it already does some math.
#  Distillation then teaches it our teacher's REASONING STYLE.
#
#  This is why DeepSeek's 7B distilled model beats GPT-4o on some
#  benchmarks — the student base was already strong.
# ═══════════════════════════════════════════════════════════════════

import gc

# Load a fresh Qwen2.5-0.5B as the student
# (teacher = our GRPO-trained + RST model, also Qwen2.5-0.5B + LoRA)
print("Loading pretrained student (Qwen2.5-0.5B, no LoRA)...")

# Free reference model from GRPO trainer to make room
del trainer2.ref
gc.collect(); torch.cuda.empty_cache()

pretrained_student = AutoModelForCausalLM.from_pretrained(
    BASE_ID, torch_dtype=torch.float16, device_map="auto"
)
# Apply small LoRA to student too (for memory efficiency during training)
from peft import get_peft_model, LoraConfig, TaskType
student_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"], lora_dropout=0.0, bias="none"
)
pretrained_student = get_peft_model(pretrained_student, student_lora)
pretrained_student.print_trainable_parameters()


def run_hf_distillation(teacher, student, tok, dataset,
                         n_steps=300, lr=3e-4, log_every=30):
    """Distil teacher into HF pretrained student."""
    dev_t  = next(teacher.parameters()).device
    dev_s  = next(student.parameters()).device
    stop   = StoppingCriteriaList([StopOnAnswerClose(tok)])
    opt    = torch.optim.AdamW(
        [p for p in student.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    it     = iter(loader)
    log    = []

    print(f"\n{'─'*56}")
    print(f"  Pretrained Student Distillation  |  {n_steps} steps")
    print(f"  Teacher: GRPO+RST Qwen → Student: Fresh Qwen")
    print(f"{'─'*56}\n")

    for step in range(1, n_steps + 1):
        try:   batch = next(it)
        except StopIteration:
            it = iter(loader); batch = next(it)

        answer = batch['correct_answer'][0]
        if not answer: continue

        prompt_ids = batch['input_ids'].to(dev_t)

        # Teacher generates trajectory
        teacher.eval()
        with torch.no_grad():
            traj = teacher.generate(
                prompt_ids, max_new_tokens=120, do_sample=False,
                pad_token_id=tok.eos_token_id, stopping_criteria=stop,
            )
            t_logits = teacher(input_ids=traj).logits.float()

        # Student learns
        student.train()
        traj_s   = traj.to(dev_s)
        s_logits = student(input_ids=traj_s).logits.float()

        # Labels: only supervise generated tokens
        labels = traj_s.clone()
        labels[:, :prompt_ids.shape[1]] = -100

        t_logits_s = t_logits.to(dev_s)

        # Trim vocab if needed (handle any mismatch)
        V_s = s_logits.shape[-1]
        V_t = t_logits_s.shape[-1]
        if V_t != V_s:
            t_logits_s = t_logits_s[:, :, :V_s]

        loss = distillation_loss_fn(s_logits, t_logits_s, labels)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in student.parameters() if p.requires_grad], 1.0
        )
        opt.step()
        log.append(loss.item())

        if step % log_every == 0:
            avg = sum(log[-log_every:]) / log_every
            print(f"  step {step:4d}/{n_steps}  distill_loss {avg:.4f}")

    print(f"\n✅ Pretrained student distillation complete")
    return student


pretrained_student = run_hf_distillation(
    teacher = model,
    student = pretrained_student,
    tok     = tokenizer,
    dataset = train_dataset,
    n_steps = 300,
    lr      = 3e-4,
    log_every = 30,
)

# ── Final evaluation: Teacher vs Student ─────────────────────────────
print("\n" + "═"*58)
print("  FINAL: TEACHER vs PRETRAINED STUDENT")
print("═"*58)

eval_qs = [
    ("Natalia sold 48 clips in April and half as many in May. Total?", "72"),
    ("6 boxes, 12 apples each. Sell 20. Remaining?",                   "52"),
    ("Tom $50, buys 3 books at $8 each. How much left?",               "26"),
    ("Train travels 60 mph for 2.5 hours. Miles travelled?",           "150"),
]

dev_s = next(pretrained_student.parameters()).device
teacher_correct = student_correct = 0

for q, gold in eval_qs:
    # Teacher
    t_resp = smart_generate(make_prompt(q))
    t_pred = extract_model_answer(t_resp)

    # Student
    pretrained_student.eval()
    ids = tokenizer(make_prompt(q), return_tensors='pt').input_ids.to(dev_s)
    stop = StoppingCriteriaList([StopOnAnswerClose(tokenizer)])
    with torch.no_grad():
        gen = pretrained_student.generate(
            ids, max_new_tokens=150, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, stopping_criteria=stop,
        )
    s_resp = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
    s_pred = extract_model_answer(s_resp)

    t_ok = t_pred == gold; s_ok = s_pred == gold
    if t_ok: teacher_correct += 1
    if s_ok: student_correct += 1

    print(f"\nQ: {q}")
    print(f"  Teacher {'✅' if t_ok else '❌'}  → {t_pred}  |  "
          f"Student {'✅' if s_ok else '❌'}  → {s_pred}  |  Gold: {gold}")

print(f"\n{'─'*58}")
print(f"  Teacher (496M GRPO+RST) : {teacher_correct}/{len(eval_qs)} = "
      f"{100*teacher_correct/len(eval_qs):.0f}%")
print(f"  Student (pretrained)    : {student_correct}/{len(eval_qs)} = "
      f"{100*student_correct/len(eval_qs):.0f}%")
print(f"{'─'*58}")

Loading pretrained student (Qwen2.5-0.5B, no LoRA)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


NameError: name 'StoppingCriteriaList' is not defined

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 12 — Phase 3: Knowledge Distillation
#
#  Teacher : the GRPO-trained `model` above (or DeepSeek-R1)
#  Student : a 4× smaller ERD model (same architecture, fewer layers)
#
#  Three loss signals:
#    α   × KL(soft_teacher ‖ soft_student)  — soft logit matching
#  (1−α) × CE(student, hard_labels)         — ground-truth anchor
#
#  The temperature T≥3 softens the teacher distribution, spreading
#  gradient signal beyond the argmax and into near-correct tokens.
# ═══════════════════════════════════════════════════════════════════

def distillation_loss(
    student_logits : torch.Tensor,   # (B, T, V)
    teacher_logits : torch.Tensor,   # (B, T, V)
    labels         : torch.Tensor,   # (B, T)  ground truth
    temperature    : float = 3.0,
    alpha          : float = 0.7,    # weight on KD term
) -> torch.Tensor:
    B, T, V = student_logits.shape
    sl = student_logits[:, :-1].contiguous().view(-1, V)
    tl = teacher_logits[:, :-1].contiguous().view(-1, V)
    y  = labels[:, 1:].contiguous().view(-1)

    # Hard cross-entropy (ground truth)
    ce_loss = F.cross_entropy(sl, y, ignore_index=-100)

    # Soft KL-divergence with temperature scaling
    s_soft  = F.log_softmax(sl / temperature, dim=-1)
    t_soft  = F.softmax(   tl / temperature, dim=-1)
    kd_loss = F.kl_div(s_soft, t_soft, reduction="batchmean") * (temperature ** 2)

    return alpha * kd_loss + (1 - alpha) * ce_loss


def build_student(teacher_cfg: ERDConfig) -> ERDModel:
    """Construct a ~4× smaller student using the same ERD architecture."""
    student_cfg = ERDConfig(
        vocab_size   = teacher_cfg.vocab_size,
        max_seq_len  = teacher_cfg.max_seq_len,
        d_model      = 256,            # 2× narrower
        n_layers     = 4,              # 2× shallower
        n_heads      = 4,
        head_dim     = 64,
        kv_lora_rank = 64,
        q_lora_rank  = 128,
        n_routed_exp = 4,
        n_shared_exp = 1,
        top_k        = 2,
        d_ff_routed  = 512,
        d_ff_shared  = 256,
    )
    return ERDModel(student_cfg)


def distill_step(
    teacher     : ERDModel,
    student     : ERDModel,
    student_opt : torch.optim.Optimizer,
    input_ids   : torch.Tensor,
    labels      : torch.Tensor,
    scaler      : torch.cuda.amp.GradScaler,
) -> float:
    teacher.eval()
    student.train()

    with torch.no_grad():
        t_logits = teacher(input_ids)["logits"]

    with torch.cuda.amp.autocast(enabled=USE_AMP):
        s_logits = student(input_ids)["logits"]
        loss     = distillation_loss(s_logits, t_logits, labels)

    student_opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.unscale_(student_opt)
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    scaler.step(student_opt)
    scaler.update()
    return loss.item()


# ── Instantiate student ───────────────────────────────────────────────
student     = build_student(config).to(device)
s_total, s_active = student.count_params()
print(f"Student : {s_total/1e6:.1f} M total / {s_active/1e6:.1f} M active")

student_opt = torch.optim.AdamW(student.parameters(), lr=1e-4, weight_decay=0.01)
scaler      = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("\nPhase 3 ready.")
print("Usage: distill_step(teacher=model, student=student, ...)")
print("Train student on teacher's <think> trajectories to get your edge model.")

KV-cache compression : 64/512 → 88% savings
FFN active compute   : top-2/4 → 50%
Student : 16.0 M total / 12.9 M active

Phase 3 ready.
Usage: distill_step(teacher=model, student=student, ...)
Train student on teacher's <think> trajectories to get your edge model.


/tmp/ipykernel_3198/1935414481.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler      = torch.cuda.amp.GradScaler(enabled=USE_AMP)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 13 — Fixed Inference with StopCriteria + Accuracy Evaluation
# ═══════════════════════════════════════════════════════════════════

from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnAnswerClose(StoppingCriteria):
    """
    Decode last 20 tokens to a string and check for </answer>.
    Robust to any tokenization of </answer> (1 token or many).
    """
    def __init__(self, tok):
        self.tok      = tok
        self.stop_str = "</answer>"

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        recent = self.tok.decode(
            input_ids[0, -20:], skip_special_tokens=False
        )
        return self.stop_str in recent


def smart_generate(prompt_text: str, max_new_tokens: int = 200) -> str:
    """Generate with proper stopping so the model doesn't loop."""
    model.eval()
    dev  = next(model.parameters()).device
    ids  = tokenizer(prompt_text, return_tensors='pt').input_ids.to(dev)
    stop = StoppingCriteriaList([StopOnAnswerClose(tokenizer)])

    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens    = max_new_tokens,
            do_sample         = False,        # greedy → deterministic
            pad_token_id      = tokenizer.eos_token_id,
            stopping_criteria = stop,
        )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


# ── Test on the two demo problems ────────────────────────────────────
demo_qs = [
    ("Natalia sold 48 clips in April and half as many in May. How many total?",
     "72"),
    ("A store has 6 boxes. Each has 12 apples. They sell 20. How many remain?",
     "52"),
    ("Tom has $50. He buys 3 books at $8 each. How much money does he have left?",
     "26"),
    ("A train travels 60 mph for 2.5 hours. How many miles does it travel?",
     "150"),
]

print("═"*58)
print("  INFERENCE EVALUATION (greedy, stop at </answer>)")
print("═"*58)
correct = 0
for q, gold in demo_qs:
    response = smart_generate(make_prompt(q))
    pred     = extract_model_answer(response)
    status   = "✅" if pred == gold else "❌"
    print(f"\n{status} Q: {q}")
    print(f"   Response : {response.strip()}")
    print(f"   Parsed   : {pred}  |  Gold: {gold}")
    if pred == gold:
        correct += 1

print(f"\n{'─'*58}")
print(f"  Score: {correct}/{len(demo_qs)} = {100*correct/len(demo_qs):.0f}%")
print(f"{'─'*58}")

══════════════════════════════════════════════════════════
  INFERENCE EVALUATION (greedy, stop at </answer>)
══════════════════════════════════════════════════════════

✅ Q: Natalia sold 48 clips in April and half as many in May. How many total?
   Response : April: 48 clips
May: 48/2 = 24 clips
Total: 48+24 = 72 clips</think><answer>72</answer>
   Parsed   : 72  |  Gold: 72

✅ Q: A store has 6 boxes. Each has 12 apples. They sell 20. How many remain?
   Response : 6*12=72 apples
72-20=52 apples</think><answer>52</answer>
   Parsed   : 52  |  Gold: 52

✅ Q: Tom has $50. He buys 3 books at $8 each. How much money does he have left?
   Response : He spent 3*8=$24
So he has 50-24=$26</think><answer>26</answer>
   Parsed   : 26  |  Gold: 26

✅ Q: A train travels 60 mph for 2.5 hours. How many miles does it travel?
   Response : It travels 150 miles</think><answer>150</answer>
   Parsed   : 150  |  Gold: 150

──────────────────────────────────────────────────────────
  Score: 4/4 = 100%
──

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 14 — Phase 3: Knowledge Distillation (FIXED)
# ═══════════════════════════════════════════════════════════════════
import torch.nn.functional as F

dev = next(model.parameters()).device

# ── FIX: Rebuild student with correct vocab size ─────────────────────
# Old student had vocab_size=32000 (ERDConfig default).
# Teacher (Qwen) generates token IDs up to 151,936 → embedding crash.
# Solution: rebuild student matching tokenizer.vocab_size exactly.

student_cfg = ERDConfig(
    vocab_size   = 151936,   # Qwen2.5-0.5B model vocab (tokenizer=151643 + 293 special tokens)
    max_seq_len  = 1024,
    d_model      = 256,
    n_layers     = 4,
    n_heads      = 4,
    head_dim     = 64,
    kv_lora_rank = 64,
    q_lora_rank  = 128,
    n_routed_exp = 4,
    n_shared_exp = 1,
    top_k        = 2,
    d_ff_routed  = 512,
    d_ff_shared  = 256,
)
student = ERDModel(student_cfg).to(dev)
total, active = student.count_params()
print(f"✅ Student rebuilt: vocab={tokenizer.vocab_size}, "
      f"{total/1e6:.1f}M params / {active/1e6:.1f}M active")


# ── Distillation loss ─────────────────────────────────────────────────
def distillation_loss_fn(
    s_logits   : torch.Tensor,
    t_logits   : torch.Tensor,
    labels     : torch.Tensor,
    temperature: float = 3.0,
    alpha      : float = 0.7,
) -> torch.Tensor:
    B, T, V = s_logits.shape
    sl = s_logits[:, :-1].reshape(-1, V)
    tl = t_logits[:, :-1].reshape(-1, V)
    y  = labels[:, 1:].reshape(-1)

    ce_loss = F.cross_entropy(sl, y, ignore_index=-100)
    s_soft  = F.log_softmax(sl / temperature, dim=-1)
    t_soft  = F.softmax(   tl / temperature, dim=-1)
    kd_loss = F.kl_div(s_soft, t_soft, reduction="batchmean") * (temperature ** 2)
    return alpha * kd_loss + (1 - alpha) * ce_loss


# ── Distillation training loop ────────────────────────────────────────
def run_distillation(teacher, student, tokenizer, dataset,
                     n_steps=150, lr=5e-4, log_every=15):

    teacher_dev = next(teacher.parameters()).device
    student_dev = next(student.parameters()).device

    opt    = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=0.01)
    loader = DataLoader(dataset, batch_size=1, shuffle=True)
    it     = iter(loader)
    stop   = StoppingCriteriaList([StopOnAnswerClose(tokenizer)])

    print(f"\n{'─'*56}")
    print(f"  Phase 3 — Distillation  |  {n_steps} steps")
    t_n = sum(p.numel() for p in teacher.parameters()) / 1e6
    s_n = sum(p.numel() for p in student.parameters()) / 1e6
    print(f"  Teacher {t_n:.0f}M  →  Student {s_n:.1f}M")
    print(f"{'─'*56}\n")

    history = []

    for step in range(1, n_steps + 1):
        try:   batch = next(it)
        except StopIteration:
            it = iter(loader); batch = next(it)

        answer = batch['correct_answer'][0]
        if not answer:
            continue

        prompt_ids = batch['input_ids'].to(teacher_dev)

        # ── Teacher generates full <think>…</think><answer>N</answer> ─
        teacher.eval()
        with torch.no_grad():
            trajectory = teacher.generate(
                prompt_ids,
                max_new_tokens    = 120,
                do_sample         = False,
                pad_token_id      = tokenizer.eos_token_id,
                stopping_criteria = stop,
            )                                        # (1, L)

            t_logits = teacher(
                input_ids = trajectory
            ).logits.float()                         # (1, L, V)  fp32

        # ── Student learns to mimic teacher trajectory ────────────────
        student.train()
        traj_s   = trajectory.to(student_dev)
        s_out    = student(traj_s)
        s_logits = s_out["logits"].float()           # (1, L, V)  fp32

        # Mask the prompt tokens — only supervise the generated part
        labels = traj_s.clone()
        labels[:, : prompt_ids.shape[1]] = -100

        t_logits_s = t_logits.to(student_dev)
        loss       = distillation_loss_fn(s_logits, t_logits_s, labels)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()

        history.append(loss.item())

        if step % log_every == 0:
            avg = sum(history[-log_every:]) / log_every
            print(f"  step {step:4d}/{n_steps}  distill_loss {avg:.4f}")

    print(f"\n✅ Distillation complete.")
    return student


# ── Run ───────────────────────────────────────────────────────────────
student = run_distillation(
    teacher   = model,
    student   = student,
    tokenizer = tokenizer,
    dataset   = GSM8KDataset(train_data, tokenizer),
    n_steps   = 150,
    lr        = 5e-4,
    log_every = 15,
)


# ── Student inference test ────────────────────────────────────────────
print("\n" + "═"*56)
print("  STUDENT MODEL TEST (16M params)")
print("═"*56)

student.eval()
test_questions = [
    ("Tom has $50. He buys 3 books at $8 each. How much left?", "26"),
    ("A store has 6 boxes, 12 apples each. They sell 20. Remaining?", "52"),
]

for q, gold in test_questions:
    ids = tokenizer(make_prompt(q), return_tensors='pt').input_ids.to(dev)
    with torch.no_grad():
        gen = student.generate(
            ids,
            max_new_tokens = 100,
            temperature    = 0.7,
            eos_token_id   = tokenizer.eos_token_id,
        )
    response = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
    pred     = extract_model_answer(response)
    status   = "✅" if pred == gold else "❌"
    print(f"\n{status}  Q: {q}")
    print(f"   Student: {response.strip()[:120]}")
    print(f"   Parsed: {pred}  |  Gold: {gold}")

KV-cache compression : 64/512 → 88% savings
FFN active compute   : top-2/4 → 50%
✅ Student rebuilt: vocab=151643, 46.7M params / 43.6M active

────────────────────────────────────────────────────────
  Phase 3 — Distillation  |  150 steps
  Teacher 496M  →  Student 46.7M
────────────────────────────────────────────────────────



KeyboardInterrupt: 

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 15 — Continue Distillation (500 more steps) + Student Eval
# ═══════════════════════════════════════════════════════════════════

# ── Continue distillation from where we left off ─────────────────────
print("Continuing distillation for 500 more steps...\n")

student = run_distillation(
    teacher   = model,
    student   = student,
    tokenizer = tokenizer,
    dataset   = GSM8KDataset(train_data, tokenizer),
    n_steps   = 500,
    lr        = 2e-4,       # slightly lower LR for continued training
    log_every = 50,
)


# ── Fixed student inference (post-process to stop at </answer>) ───────
def student_infer(question: str, temperature: float = 0.3) -> str:
    """
    Run the student model and cleanly cut output at </answer>.
    Lower temperature (0.3) keeps the undertrained student more stable.
    """
    student.eval()
    ids = tokenizer(
        make_prompt(question), return_tensors='pt'
    ).input_ids.to(dev)

    with torch.no_grad():
        gen = student.generate(
            ids,
            max_new_tokens = 150,
            temperature    = temperature,
            top_p          = 0.9,
            eos_token_id   = tokenizer.eos_token_id,
        )

    text = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)

    # Hard-stop at first </answer> — avoids looping
    if "</answer>" in text:
        text = text[: text.index("</answer>") + len("</answer>")]

    return text.strip()


# ── Evaluate student on 4 problems ───────────────────────────────────
eval_problems = [
    ("Natalia sold 48 clips in April and half as many in May. How many total?", "72"),
    ("A store has 6 boxes. Each has 12 apples. They sell 20. How many remain?",  "52"),
    ("Tom has $50. He buys 3 books at $8 each. How much does he have left?",     "26"),
    ("A train travels 60 mph for 2.5 hours. How many miles does it travel?",     "150"),
]

print("\n" + "═"*58)
print(f"  STUDENT TEST  ({student.count_params()[0]/1e6:.0f}M params)")
print("═"*58)

correct = 0
for q, gold in eval_problems:
    response = student_infer(q)
    pred     = extract_model_answer(response)
    status   = "✅" if pred == gold else "❌"
    print(f"\n{status}  Q: {q}")
    print(f"   {response}")
    print(f"   Parsed: {pred}  |  Gold: {gold}")
    if pred == gold:
        correct += 1

print(f"\n{'─'*58}")
print(f"  Student score : {correct}/{len(eval_problems)} = "
      f"{100*correct/len(eval_problems):.0f}%")
print(f"  Teacher score : 2/4 = 50%  (from Cell 13)")
print(f"{'─'*58}")

Continuing distillation for 500 more steps...


────────────────────────────────────────────────────────
  Phase 3 — Distillation  |  500 steps
  Teacher 496M  →  Student 46.7M
────────────────────────────────────────────────────────

  step   50/500  distill_loss 2.9305
  step  100/500  distill_loss 2.9924
  step  150/500  distill_loss 2.8483
  step  200/500  distill_loss 2.6751
  step  250/500  distill_loss 2.5240
  step  300/500  distill_loss 2.5264
  step  350/500  distill_loss 2.5811
  step  400/500  distill_loss 2.5645
  step  450/500  distill_loss 2.4849
  step  500/500  distill_loss 2.4718

✅ Distillation complete.

══════════════════════════════════════════════════════════
  STUDENT TEST  (47M params)
══════════════════════════════════════════════════════════

❌  Q: Natalia sold 48 clips in April and half as many in May. How many total?
   2</think>
<answer> 2</answer>
   Parsed: 2  |  Gold: 72

❌  Q: A store has 6 boxes. Each has 12 apples. They sell 20. How many remain?
   2

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELL 16 — Save Trained Models + Print Final Pipeline Report
# ═══════════════════════════════════════════════════════════════════

import os, json
from datetime import datetime

SAVE_DIR = "/content/erd_pipeline_output"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Save teacher (GRPO-trained Qwen + LoRA) ───────────────────────────
print("Saving teacher (LoRA weights only)...")
model.save_pretrained(f"{SAVE_DIR}/teacher_lora")
tokenizer.save_pretrained(f"{SAVE_DIR}/teacher_lora")
print(f"✅ Teacher saved → {SAVE_DIR}/teacher_lora")

# ── Save student (ERDModel weights) ──────────────────────────────────
print("\nSaving student ERDModel...")
torch.save({
    'model_state_dict' : student.state_dict(),
    'config'           : student.cfg,
    'distill_steps'    : 650,
    'final_loss'       : 2.4718,
}, f"{SAVE_DIR}/student_erd.pt")
print(f"✅ Student saved → {SAVE_DIR}/student_erd.pt")

# ── Final pipeline report ─────────────────────────────────────────────
report = {
    "pipeline"   : "Efficient Reasoning & Distillation (ERD)",
    "timestamp"  : datetime.now().isoformat(),
    "phase1_architecture" : {
        "backbone"          : "Sparse MLA + MoE Transformer",
        "total_params"      : "190M",
        "active_params"     : "115M",
        "kv_compression"    : "87.5%",
        "ffn_active"        : "25% (top-2 of 8 experts)",
    },
    "phase2_grpo" : {
        "base_model"        : "Qwen/Qwen2.5-0.5B",
        "algorithm"         : "GRPO + LoRA (r=16)",
        "steps"             : 200,
        "reward_start"      : -0.48,
        "reward_peak"       : +0.34,
        "teacher_accuracy"  : "50% on GSM8K (4 problems)",
        "human_labels"      : 0,
    },
    "phase3_distillation" : {
        "teacher_params"    : "496M",
        "student_params"    : "47M",
        "compression_ratio" : "10.6×",
        "steps"             : 650,
        "loss_start"        : 8.45,
        "loss_final"        : 2.47,
    },
    "to_reach_production_quality" : [
        "Start student from pretrained checkpoint (e.g. Qwen2.5-0.5B)",
        "Run GRPO for 2000+ steps on teacher",
        "Run distillation for 50,000+ steps",
        "Use multi-GPU setup (8× A100)",
    ]
}

report_path = f"{SAVE_DIR}/pipeline_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print("\n" + "═"*58)
print("  ERD PIPELINE — FINAL REPORT")
print("═"*58)
for phase, details in report.items():
    if isinstance(details, dict):
        print(f"\n  {phase.upper().replace('_', ' ')}")
        for k, v in details.items():
            print(f"    {k:<22} : {v}")
    elif isinstance(details, list):
        print(f"\n  {phase.upper().replace('_', ' ')}")
        for item in details:
            print(f"    → {item}")

print(f"\n✅ All outputs saved to {SAVE_DIR}")
print("   Download via Files panel (left sidebar in Colab)")
print("═"*58)

Saving teacher (LoRA weights only)...
✅ Teacher saved → /content/erd_pipeline_output/teacher_lora

Saving student ERDModel...
✅ Student saved → /content/erd_pipeline_output/student_erd.pt

══════════════════════════════════════════════════════════
  ERD PIPELINE — FINAL REPORT
══════════════════════════════════════════════════════════

  PHASE1 ARCHITECTURE
    backbone               : Sparse MLA + MoE Transformer
    total_params           : 190M
    active_params          : 115M
    kv_compression         : 87.5%
    ffn_active             : 25% (top-2 of 8 experts)

  PHASE2 GRPO
    base_model             : Qwen/Qwen2.5-0.5B
    algorithm              : GRPO + LoRA (r=16)
    steps                  : 200
    reward_start           : -0.48
    reward_peak            : 0.34
    teacher_accuracy       : 50% on GSM8K (4 problems)
    human_labels           : 0

  PHASE3 DISTILLATION
    teacher_params         : 496M
    student_params         : 47M
    compression_ratio      : 10.6×
  

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  RELOAD CELL — Restores all variables lost after runtime reset
#  Run this ONCE before running any Improvement Cell
# ═══════════════════════════════════════════════════════════════════

import os, gc, re, copy, contextlib, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          StoppingCriteria, StoppingCriteriaList)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ── Tokenizer + Model ────────────────────────────────────────────────
BASE_ID   = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, torch_dtype=torch.float16, device_map="auto"
)
lora_cfg = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout   = 0.0,
    bias           = "none",
)
model = get_peft_model(model, lora_cfg)
dev   = next(model.parameters()).device

# ── Dataset ──────────────────────────────────────────────────────────
gsm8k      = load_dataset("openai/gsm8k", "main")
train_data = gsm8k["train"]

# ── Helper functions ─────────────────────────────────────────────────
SYSTEM = (
    "You are a math reasoning assistant. "
    "Show all working inside <think></think> tags, "
    "then give ONLY the final integer inside <answer></answer> tags.\n"
)

def make_prompt(q):
    return f"{SYSTEM}\nProblem: {q}\n\n<think>"

def extract_gsm8k_answer(s):
    m = re.search(r'####\s*(-?[\d,]+)', s)
    return m.group(1).replace(',', '') if m else None

def extract_model_answer(s):
    m = re.search(r'<answer>\s*(-?[\d,]+)\s*</answer>', s)
    return m.group(1).replace(',', '') if m else None

class StopOnAnswerClose(StoppingCriteria):
    def __init__(self, tok):
        self.tok = tok
    def __call__(self, input_ids, scores, **kwargs):
        recent = self.tok.decode(input_ids[0, -20:], skip_special_tokens=False)
        return "</answer>" in recent

def smart_generate(prompt_text, max_new_tokens=200):
    model.eval()
    ids  = tokenizer(prompt_text, return_tensors='pt').input_ids.to(dev)
    stop = StoppingCriteriaList([StopOnAnswerClose(tokenizer)])
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, stopping_criteria=stop,
        )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

class GSM8KDataset(Dataset):
    def __init__(self, data, tok, max_len=220):
        self.data = data; self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tok(make_prompt(item['question']),
                        max_length=self.max_len, truncation=True,
                        return_tensors='pt')
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'correct_answer' : extract_gsm8k_answer(item['answer']),
        }

train_dataset = GSM8KDataset(train_data, tokenizer)

print(f"\n✅ Reload complete")
print(f"   Model  : {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")
print(f"   Dataset: {len(train_data)} problems")
print(f"   VRAM   : {torch.cuda.memory_allocated()/2**30:.1f} GB")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Loading model...


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


✅ Reload complete
   Model  : 496M params
   Dataset: 7473 problems
   VRAM   : 1.0 GB
